### Imports

In [1]:
import os

import numpy as np
import skimage.io as io
import cv2 as cv
import random
import shutil

cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(parent_dir, 'openslide-bin-4.0.0.6-windows-x64\\bin')
# OPENSLIDE_PATH = r'C:\\Users\\Waluigi\\Desktop\\github_repos\\TILseg2\\openslide-bin-4.0.0.6-windows-x64\\bin'
print(OPENSLIDE_PATH)
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide
import xml.etree.cElementTree as ET

# need to (1) actiate environment and then (2) run 'pip install openslide-python' in command line 
# after installing the binaries for openslide to be imported properly is using WINDOWS

c:\Users\Waluigi\Desktop\github_repos\TILseg2\openslide-bin-4.0.0.6-windows-x64\bin


### Extract Patches

In [2]:
mlevel = 0  # reading the WSI at level 1 (40x)
factor = 2 ** mlevel
patch_size_48 = (48, 48)
patch_size_256 = (256, 256)
padding_percent = 0

# Set the path to the folder containing WSIs and XML annotations
# TODO
folder_path = r"E:\\Re-training Training\\annotations"

# Loop through .svs files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".svs"):
        # Extract the slide name (without extension)
        slidename = os.path.splitext(file_name)[0]

        # Construct the corresponding XML annotation file name
        annotname = f"{slidename}.xml"

        # Load the annotated WSI image using OpenSlide
        wsi_path = os.path.join(folder_path, file_name)
        wsi = openslide.OpenSlide(wsi_path)

        # Loop through each annotation ID category
        # id_folders = ['id_1_redstroma', 'id_2_necrosis', 'id_3_stromalcells', 'id_4_canadjstroma']
        # id_folders = ['epithelium_new']
        # TODO
        id_folders = ['id1_stroma', 'id2_epithelium', 'id3_other'] 
        for i, folder in enumerate(id_folders):
            if folder == 'skip': # skip layers that don't need to be extracted
                continue

            # create folder for corresponding annotation ID
            anno_id = i + 1
            id_dir = os.path.join(folder_path, folder)
            slide_dir = os.path.join(id_dir, slidename)
        
            # parse .xml file into an annotation list
            def parse_xml(anno_path, 
                          id_num):
                tree = ET.ElementTree(file=anno_path)
                annolist = {}
                root = tree.getroot()
                for annotation in root.iter('Annotation'):
                    if annotation.attrib.get('Id') == str(id_num):
                        i = 0
                        # for annotation in root.findall("Annotation"):
                        for region in annotation.findall(".//Region"):
                            vasc = []
                            for vertex in region.findall(".//Vertex"):
                                try:
                                    x = float(vertex.attrib.get("X"))
                                    y = float(vertex.attrib.get("Y"))
                                    vasc.append((int(x / factor), int(y / factor)))
                                except Exception as e:
                                    print(f"Error parsing coordinates: {e}")
                                    continue
                            annolist[i] = vasc
                            i += 1

                if len(annolist) == 0:
                    anno_filename = os.path.basename(anno_path)
                    print(f'There are no annotations for ID {id_num} in {anno_filename}')

                print(annolist)

                return annolist


            def is_mostly_white(image, 
                                threshold=0.7):
                # Convert the image to grayscale
                gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
                # Apply threshold to identify white pixels
                _, binary = cv.threshold(gray, 200, 255, cv.THRESH_BINARY)
                # Calculate the percentage of white pixels
                white_percentage = np.sum(binary == 255) / binary.size
                return white_percentage >= threshold


            def extract_patches_with_padding(annolist, 
                                             patch_size, 
                                             padding_percent=0, 
                                             overlap_percent=0):
                patches = []
                patch_width, patch_height = patch_size
                padding = int(min(patch_width, patch_height) * (padding_percent / 100))
                overlap = int(min(patch_width, patch_height) * (overlap_percent / 100))
                
                for i, coords in annolist.items():
                    x, y, w, h = cv.boundingRect(np.array(coords))
                    
                    x_start, y_start = x, y
                    x_end, y_end = x + w, y + h
                    
                    while (y_start + patch_height) <= y_end: # ensures nothing is being extracted outside of the annotation box
                        x_curr = x_start
                        while (x_curr + patch_width) <= x_end:
                            x_patch_start = max(x_curr - padding, 0)
                            y_patch_start = max(y_start - padding, 0)
                            x_patch_end = min(x_patch_start + patch_width, wsi.level_dimensions[0][0])
                            y_patch_end = min(y_patch_start + patch_height, wsi.level_dimensions[0][1])
                            
                            #patch_img = wsi.read_region((x_patch_start * factor, y_patch_start * factor), mlevel, (patch_width, patch_height))
                            patch_img_rgba = np.asarray(wsi.read_region((x_patch_start * factor, y_patch_start * factor), mlevel, (x_patch_end - x_patch_start, y_patch_end - y_patch_start)))
                            patch_img = patch_img_rgba[:, :, :3]
                            if not is_mostly_white(patch_img):
                                patches.append(patch_img)
                            
                            x_curr += patch_width - overlap
                        y_start += patch_height - overlap
                
                return patches


            def save_patches(patches, output_dir, slide_name, patch_size):
                os.makedirs(output_dir, exist_ok=True)
                patch_w, _ = patch_size
                
                for i, patch in enumerate(patches):

                    save_as = os.path.join(output_dir, f'{slide_name}_{folder}_{patch_w}_patch_position_{i}.tif')
                    io.imsave(save_as, patch, check_contrast=False)


            def generate_patches(out_dir, 
                                 slidename, 
                                 in_dir, 
                                 annotname, 
                                 patch_size, 
                                 id_num, 
                                 padding_percent=5, 
                                 overlap_percent=5):
                
                annopath = os.path.join(in_dir, annotname)
                annolist = parse_xml(annopath, id_num)
                patches = extract_patches_with_padding(annolist, patch_size, padding_percent, overlap_percent)
                
                save_patches(patches, out_dir, slidename, patch_size)


            # Call the function to generate 48x48 and 256x256 patches for the current pair of WSI and XML
            out_48_dir = os.path.join(slide_dir, '48')
            os.makedirs(out_48_dir, exist_ok=True)
            generate_patches(out_48_dir, 
                             slidename, 
                             folder_path, 
                             annotname, 
                             patch_size_48, 
                             id_num=anno_id,
                             padding_percent=0, overlap_percent=0)
            
            out_256_dir = os.path.join(slide_dir, '256')
            os.makedirs(out_256_dir, exist_ok=True)
            generate_patches(out_256_dir, 
                             slidename, 
                             folder_path, 
                             annotname, 
                             patch_size_256, 
                             id_num=anno_id,
                             padding_percent=0, overlap_percent=0)

{0: [(39578, 24692), (40602, 24692), (40602, 25796), (39578, 25796)], 1: [(40326, 26935), (40582, 26935), (40582, 27422), (40326, 27422)], 2: [(43505, 25767), (44007, 25767), (44007, 26175), (43505, 26175)], 3: [(44083, 26188), (44234, 26188), (44234, 26356), (44083, 26356)], 4: [(44426, 26283), (44551, 26283), (44551, 26407), (44426, 26407)], 5: [(44513, 26107), (44618, 26107), (44618, 26229), (44513, 26229)]}
{0: [(39578, 24692), (40602, 24692), (40602, 25796), (39578, 25796)], 1: [(40326, 26935), (40582, 26935), (40582, 27422), (40326, 27422)], 2: [(43505, 25767), (44007, 25767), (44007, 26175), (43505, 26175)], 3: [(44083, 26188), (44234, 26188), (44234, 26356), (44083, 26356)], 4: [(44426, 26283), (44551, 26283), (44551, 26407), (44426, 26407)], 5: [(44513, 26107), (44618, 26107), (44618, 26229), (44513, 26229)]}
{0: [(47202, 23141), (47759, 23141), (47759, 23417), (47202, 23417)], 1: [(47488, 23984), (48065, 23984), (48065, 24265), (47488, 24265)], 2: [(47312, 24356), (48020, 243

In [31]:
def parse_xml(anno_path):
    tree = ET.ElementTree(file=anno_path)
    annolist = {}
    root = tree.getroot()
    i = 0
    for coords in root.iter('Coordinates'):
        vasc = []
        for coord in coords:
            vasc.append((int(float(coord.attrib.get("X")) / factor), int(float(coord.attrib.get("Y")) / factor)))
        annolist[i] = vasc
        i += 1
    return annolist

### Choose patches for retraining

In [4]:
## directory tree layout required for this:

def sample_images(source_folder, 
                  test_folder,
                  size, # either 48 or 256
                  total_samples=4200):
    # Get the list of child folders (i.e. 2XXXXXX_H&E)
    child_folders = [f for f in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, f))]

    # Calculate how many images to sample from each child folder
    num_child_folders = len(child_folders)
    if num_child_folders == 0:
        print("No child folders found.")
        return

    images_per_folder = total_samples // num_child_folders
    print(f"Sampling {images_per_folder} images from each of the {num_child_folders} child folders.")

    # Ensure the destination folder exists
    os.makedirs(test_folder, exist_ok=True)

    images_moved = 0

    # Process each child folder
    for child_folder in child_folders:
        child_folder_path = os.path.join(source_folder, child_folder)
        child_folder_patch_path = os.path.join(child_folder_path, str(size))

        # Get a list of image files in the child folder
        image_files = [f for f in os.listdir(child_folder_patch_path) if os.path.isfile(os.path.join(child_folder_patch_path, f))]

        # If there are fewer files than needed, sample all of them
        images_to_sample = min(images_per_folder, len(image_files))
        images_moved += images_to_sample

        print(f"sampling {images_to_sample} patches from {child_folder}")

        # Randomly sample images
        sampled_images = random.sample(image_files, images_to_sample)

        # Copy the sampled images to the destination folder
        for image in sampled_images:
            src_image_path = os.path.join(child_folder_patch_path, image)
            test_image_path = os.path.join(test_folder, image)

            # Check if the file already exists in the destination folder to avoid overwriting
            if os.path.exists(test_image_path):
                name, ext = os.path.splitext(image)
                test_image_path = os.path.join(test_folder, f"{name}_{child_folder}{ext}")

            shutil.copy(src_image_path, test_image_path)
            # print(f"Copied {image} to {test_image_path}")

    print(f"Successfully sampled and copied {images_moved} images.")

In [8]:
# TODO
source_folder = "E:\\Re-training Training\\annotations\\id1_stroma"  
training_folder = "E:\\Re-training Training\\retraining\\stroma" 
size = 256
total_samples = 100

sample_images(source_folder, training_folder, size, total_samples)

Sampling 100 images from each of the 1 child folders.
sampling 18 patches from tp18-p69
Successfully sampled and copied 18 images.


In [12]:
2000-386

1614

### Misc

In [ ]:
            # def parse_xml(anno_path):
            #     tree = ET.ElementTree(file=anno_path)
            #     annolist = {}
            #     root = tree.getroot()
            #     i = 0
            #     for coords in root.iter('Annotation'):
            #         vasc = []
            #         for coord in coords:
            #             vasc.append((int(float(coord.attrib.get("X")) / factor), int(float(coord.attrib.get("Y")) / factor)))
            #         annolist[i] = vasc
            #         i += 1
            #     return annolist
            
            # def parse_xml(anno_path):
            #     tree = ET.ElementTree(file=anno_path)
            #     annolist = {}
            #     root = tree.getroot()
            #     for annotation in root.iter('Annotation'):
            #         if annotation.attrib.get('Id') == '2':
            #             i = 0
            #             for coords in root.iter('Coordinates'):
            #                 vasc = []
            #                 for coord in coords:
            #                     vasc.append((int(float(coord.attrib.get("X")) / factor), int(float(coord.attrib.get("Y")) / factor)))
            #                 annolist[i] = vasc
            #                 i += 1
            #     return annolist
            